In [1]:
import torch
import torch_directml

device = torch_directml.device()

x = torch.tensor([2.0], device=device, requires_grad=True)
loss = (x * x).sum()
loss.backward()

print("Device:", device)
print("Result:", loss.cpu().item())
print("Gradient:", x.grad.cpu().item())

Device: privateuseone:0
Result: 4.0
Gradient: 4.0


In [2]:
from pathlib import Path
import csv

# Find the dataset folder directly below the notebook's working folder.
candidates = [
    folder for folder in Path.cwd().iterdir()
    if folder.is_dir()
    and (folder / "train_labels.csv").exists()
    and (folder / "sample_submission.csv").exists()
]

if len(candidates) != 1:
    raise ValueError(
        f"Expected one dataset folder, found {len(candidates)}. "
        f"Working folder: {Path.cwd()}"
    )

DATA_ROOT = candidates[0]

print("Dataset folder:", DATA_ROOT)
print("Training folder exists:", (DATA_ROOT / "train").is_dir())
print("Test folder exists:", (DATA_ROOT / "test").is_dir())

for filename in ["train_labels.csv", "sample_submission.csv"]:
    with (DATA_ROOT / filename).open(
        newline="", encoding="utf-8-sig"
    ) as file:
        reader = csv.reader(file)
        header = next(reader)
        first_row = next(reader)
        row_count = 1 + sum(1 for _ in reader)

    print(f"\n{filename}")
    print("Columns:", header)
    print("First row:", first_row)
    print("Data rows:", row_count)

Dataset folder: c:\Users\Danie\OneDrive\Documents\GitHub\Kaggle-Competition-2026\enge-707-2026-s-2-cat-and-dog-facial-feature-classification-ch (1)
Training folder exists: True
Test folder exists: True

train_labels.csv
Columns: ['image_id', 'class']
First row: ['00001', 'DogEye']
Data rows: 63660

sample_submission.csv
Columns: ['image_id', 'class']
First row: ['1', 'DogEye']
Data rows: 29443


In [3]:
%pip install pandas matplotlib scikit-learn tqdm

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ------------------ --------------------- 4.5/9.9 MB 22.3 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 24.5 MB/s  0:00:00
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   ------ --------------------------------- 1.6/9.3 MB 8.3 MB/s eta 0:00:01
   ----------------------- ---------------- 5.5/9.3 MB 12.9 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 15.7 MB/s  0:00:00
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   --------------------------- ------------ 5.8/8.3 MB 29.3 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 24.6 MB/s  0:00:00


In [6]:
import time
import torch
import torch_directml
from torch import nn
from torchvision.models import efficientnet_b0

device = torch_directml.device()
print("Device:", device)

# Random weights and synthetic images: compatibility test only.
# Actual training will use pretrained weights and your labelled images.
test_model = efficientnet_b0(weights=None)
test_model.classifier[1] = nn.Linear(
    test_model.classifier[1].in_features, 8
)
test_model = test_model.to(device)
test_model.train()

optimizer = torch.optim.SGD(
    test_model.parameters(),
    lr=0.001,
    momentum=0.9,
    weight_decay=1e-4,
    foreach=False
)
criterion = nn.CrossEntropyLoss()

# Small batch to limit memory use.
images = torch.randn(2, 3, 224, 224).to(device)
targets = torch.tensor([0, 1], dtype=torch.long).to(device)

start = time.perf_counter()

for step in range(2):
    optimizer.zero_grad(set_to_none=True)

    logits = test_model(images)
    loss = criterion(logits, targets)

    loss.backward()
    optimizer.step()

    # Copying to CPU also waits for this result to finish.
    loss_value = loss.detach().cpu().item()
    print(f"Step {step + 1}: loss = {loss_value:.4f}")

print("Output shape:", tuple(logits.shape))
print(f"Elapsed: {time.perf_counter() - start:.1f} seconds")
print("Model forward pass, backpropagation and optimizer test completed.")

# Release test objects before loading the real training model.
del test_model, optimizer, images, targets, logits, loss

Device: privateuseone:0
Step 1: loss = 2.1318
Step 2: loss = 1.9953
Output shape: (2, 8)
Elapsed: 0.6 seconds
Model forward pass, backpropagation and optimizer test completed.
